In [1]:
from pathlib import Path
import re, json, time, shutil
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
ROOT = Path("E:/2026_capstone/policy_data")
assert ROOT.exists()    , f"ROOT not found: {ROOT.resolve()}"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = Path("PIPLINE_OUTPUT") / f"run_{RUN_ID}"
OUT.mkdir(parents=True, exist_ok=True)

OUT

WindowsPath('PIPLINE_OUTPUT/run_20260122_220214')

In [4]:
def infer_dept_from_path(p:Path) -> str:
    for part in p.parts:
        if part.endswith("_data"):
            return part.replace("_data", "")
    return "unknown"

def scan_corpus(root:Path) -> pd.DataFrame:
    rows = []
    for txt_path in root.glob("*_data/raw_text/*.txt"):
        dept = infer_dept_from_path(txt_path)
        base = txt_path.stem
        meta_path = txt_path.parents[1] / "meta" / f"{base}.json"
        xml_path = txt_path.parents[1] / "xml" / f"{base}.xml"

        rows.append({
            "dept": dept,
            "base": base,
            "txt_path": txt_path,
            "meta_path": meta_path if meta_path.exists() else None,
            "xml_path": xml_path if xml_path.exists() else None,
            "txt_size": txt_path.stat().st_size
        })
    
    df = pd.DataFrame(rows).sort_values(["dept", "base"]).reset_index(drop=True)
    return df

df = scan_corpus(ROOT)
df.head(), df.shape

(     dept                   base  \
 0  policy  01_02_2020_2019-28306   
 1  policy  01_03_2013_2012-30634   
 2  policy  01_03_2014_2013-31257   
 3  policy  01_03_2017_2016-31380   
 4  policy  01_03_2018_2017-28386   
 
                                             txt_path  \
 0  E:\2026_capstone\policy_data\doe_data\raw_text...   
 1  E:\2026_capstone\policy_data\doi_data\raw_text...   
 2  E:\2026_capstone\policy_data\doe_data\raw_text...   
 3  E:\2026_capstone\policy_data\fema_data\raw_tex...   
 4  E:\2026_capstone\policy_data\doi_data\raw_text...   
 
                                            meta_path  \
 0  E:\2026_capstone\policy_data\doe_data\meta\01_...   
 1  E:\2026_capstone\policy_data\doi_data\meta\01_...   
 2  E:\2026_capstone\policy_data\doe_data\meta\01_...   
 3  E:\2026_capstone\policy_data\fema_data\meta\01...   
 4  E:\2026_capstone\policy_data\doi_data\meta\01_...   
 
                                             xml_path  txt_size  
 0  E:\2026_capstone\p

In [ ]:
CONFIG = {
    # 1) 核心 tribal 词
    "tribal_core_terms": [
        r"\btribal\b",
        r"\btribe(s)?\b",
        r"\btribal nation(s)?\b",
        r"\bindian country\b",
        r"\breservation(s)?\b",
        r"\bsovereign(ty)?\b",
        r"\bself[-\s]?determination\b",
        r"\btrust land(s)?\b",
        r"\bBIA\b", r"\bBureau of Indian Affairs\b",
        r"\bIHS\b", r"\bIndian Health Service\b",
        r"\bTHPO\b", r"\bTribal Historic Preservation\b",
    ],
    # 2) climate/adaptation 词
    "climate_terms": [
        r"\bclimate change\b",
        r"\bclimate adaptation\b",
        r"\bresilien(ce|t)\b",
        r"\bhazard mitigation\b",
        r"\bdrought\b",
        r"\bflood(ing)?\b",
        r"\bheat\b",
        r"\bwildfire(s)?\b",
        r"\bsea level\b",
    ],
    # 3) 否分项
    "procedural_patterns": [
        r"\bfederal,\s*tribal,\s*state,\s*(and\s*)?local\b",
        r"\bfederal,\s*state,\s*local,\s*(and\s*)?tribal\b",
        r"\bfederal,\s*state,\s*tribal,\s*(and\s*)?local\b",
        r"\bpermits?,\s*approvals?,\s*(and\s*)?other legal authority\b.*\btribal\b",
    ],
    # 权重
    "weights": {
        "core_hits": 1.0,            # core 词命中
        "climate_hits": 0.4,          # climate 词命中
        "lead_bonus": 1.2,            # 前段加权
        "meta_bonus": 1.0,            # meta 标题/agency 命中加分
        "procedural_penalty": 2.0,    # 程序性提及扣分
        "length_norm": 0.15,          # 长文轻微归一化
    },
    # lead 区域长度
    "lead_chars": 1500,
}

In [6]:
def _count_patterns(text:str, patterns) -> int:
    n = 0
    for pat in patterns:
        n+= len(re.findall(pat, text, flags=re.IGNORECASE))
    return n

def read_text(p: Path, max_chars = None) -> str:
    try:
        t = p.read_text(encoding="utf-8", errors="replace")
    except Exception: 
        t = p.read_text(errors="ignore")
    if max_chars:
        t = t[:max_chars]
    return t

def read_meta(meta_path) -> dict:
    if not meta_path:
        return
    try:
        return json.loads(Path(meta_path).read_text(encoding="utf-8"))
    except Exception:
        return {}
    
def score_doc(txt_path: Path, meta_path:None, config=CONFIG):
    text = read_text(txt_path)
    lead = text[:config["lead_chars"]]
    meta = read_meta(meta_path)

    core_all = _count_patterns(text, config["tribal_core_terms"])
    core_lead = _count_patterns(lead, config["tribal_core_terms"])

    climate_all = _count_patterns(text, config["climate_terms"])
    climate_lead = _count_patterns(lead, config["climate_terms"])

    procedural = _count_patterns(text, config["procedural_patterns"])

    meta_text = " ".join([
        str(meta.get("title","")),
        str(meta.get("agency_names","")),
        str(meta.get("type","")),
        str(meta.get("citation","")),
    ])
    meta_hits = _count_patterns(meta_text, config["tribal_core_terms"])

    w = config["weights"]

    # 基础分
    raw = (
        w["core_hits"] * core_all +
        w["climate_hits"] * climate_all +
        w["lead_bonus"] * (core_lead + 0.6 * climate_lead) +
        w["meta_bonus"] * meta_hits
        - w["procedural_penalty"] * procedural
    )

    length = max(len(text), 1)
    raw = raw / (1+ w["length_norm"] * np.log10(length))

    # 压缩到0-1
    score = 1 / (1 + np.exp(-raw/3.0))

    features = {
        "core_all": core_all,
        "core_lead": core_lead,
        "climate_all": climate_all,
        "climate_lead": climate_lead,
        "procedural": procedural,
        "meta_hits": meta_hits,
        "raw": float(raw),
        "score": float(score),
    }
    return score, features

In [ ]:
scores = []
for _, r in df.iterrows():
    s, feat = score_doc(r["txt_path"], r["meta_path"])
    scores.append({
        **r.to_dict(),
        **feat
    })

In [ ]:
scored = pd.DataFrame(scores)
scored["dept"] = scored["dept"].astype(str)
scored.sort_values(["dept", "score"], ascending=[True, False], inplace=True)

scores_csv = OUT / "scores.csv"
scored.to_csv(scores_csv, index=False)

,dept,base,txt_path,meta_path,xml_path,txt_size,core_all,core_lead,climate_all,climate_lead,procedural,meta_hits,raw,score
1,policy,01_03_2013_2012-30634,E:\2026_capstone\policy_data\doi_data\raw_text...,E:\2026_capstone\policy_data\doi_data\meta\01_...,E:\2026_capstone\policy_data\doi_data\xml\01_0...,1238801,1054,0,196,0,0,0,591.970058,1.000000
12,policy,01_05_2017_2016-30004,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\01_...,E:\2026_capstone\policy_data\doe_data\xml\01_0...,1072235,5,0,1004,3,0,0,214.750804,1.000000
58,policy,01_15_2025_2025-00709,E:\2026_capstone\policy_data\doi_data\raw_text...,E:\2026_capstone\policy_data\doi_data\meta\01_...,E:\2026_capstone\policy_data\doi_data\xml\01_1...,153525,221,5,1,0,0,1,128.549111,1.000000
68,policy,01_18_2023_2022-28595,E:\2026_capstone\policy_data\epa_data\raw_text...,E:\2026_capstone\policy_data\epa_data\meta\01_...,E:\2026_capstone\policy_data\epa_data\xml\01_1...,1144916,231,0,151,0,1,0,151.692358,1.000000
72,policy,01_19_2021_2021-00967,E:\2026_capstone\policy_data\usda_data\raw_tex...,E:\2026_capstone\policy_data\usda_data\meta\01...,E:\2026_capstone\policy_data\usda_data\xml\01_...,673689,1109,0,15,0,3,0,592.027565,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
674,policy,07_19_2006_E6-11431,E:\2026_capstone\policy_data\usda_data\raw_tex...,E:\2026_capstone\policy_data\usda_data\meta\07...,E:\2026_capstone\policy_data\usda_data\xml\07_...,31838,0,0,2,0,0,0,0.477847,0.539737
192,policy,03_04_2011_2011-4878,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\03_...,E:\2026_capstone\policy_data\doe_data\xml\03_0...,16640,2,0,1,0,1,0,0.245097,0.520413
181,policy,02_29_2024_2024-04268,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\02_...,E:\2026_capstone\policy_data\doe_data\xml\02_2...,8441,2,0,0,0,1,0,0.000000,0.500000
1037,policy,11_08_2019_2019-24372,E:\2026_capstone\policy_data\fema_data\raw_tex...,E:\2026_capstone\policy_data\fema_data\meta\11...,E:\2026_capstone\policy_data\fema_data\xml\11_...,9857,1,0,1,0,1,0,-0.375505,0.468749


In [ ]:
# 改dept
def parse_dept_from_path(path_str: str) -> str:
    if pd.isna(path_str):
        return "unknown"

    # 统一分隔符
    p = str(path_str).replace("\\", "/")
    parts = [x for x in p.split("/") if x]  # 去空段

    # 1) 优先：找到 policy_data 后面的那个 *_data
    if "policy_data" in parts:
        i = parts.index("policy_data")
        if i + 1 < len(parts):
            nxt = parts[i + 1]  # 例如 doe_data
            m = re.match(r"([A-Za-z0-9_-]+)_data$", nxt, flags=re.IGNORECASE)
            if m:
                return m.group(1).lower()

    # 2) 兜底：在所有分段里找 *_data（但排除 policy_data 本身）
    for seg in parts:
        if seg.lower() == "policy_data":
            continue
        m = re.match(r"([A-Za-z0-9_-]+)_data$", seg, flags=re.IGNORECASE)
        if m:
            return m.group(1).lower()

    return "unknown"


# 假设你的 dataframe 叫 scored（或 df）
# 你图里列名是 dept / base / txt_path / ... / score or raw
scored["dept"] = scored["txt_path"].apply(parse_dept_from_path)

scored["dept"].value_counts()

dept
doi     408
usda    269
doe     262
epa     142
fema    124
hud      34
Name: count, dtype: int64

In [ ]:
# 使用minmax scaler 归一化 raw 分数，基于部门
def minmax_norm_by_group(df: pd.DataFrame, group_col: str, value_col: str, out_col: str):
    df = df.copy()
    def _mm(s):
        lo, hi = float(s.min()), float(s.max())
        if hi <= lo:
            return pd.Series(np.zeros(len(s)), index=s.index)
        return (s - lo) / (hi - lo)
    df[out_col] = df.groupby(group_col)[value_col].transform(_mm)
    return df

scored = minmax_norm_by_group(scored, group_col="dept", value_col="raw", out_col="score_norm")
scored[["dept","raw","score_norm"]].head()


,dept,raw,score_norm
1,doi,591.970058,0.668842
12,doe,214.750804,0.941995
58,doi,128.549111,0.143652
68,epa,151.692358,0.235459
72,usda,592.027565,1.000000


In [ ]:
#计算percentile，基于percentile确定保留与否，基于部门
scored["pct"] = scored.groupby("dept")["score_norm"].rank(pct=True)

scored["tier"] = np.select(
    [
        scored["pct"] >= 0.85,    # Top 15%：强相关
        scored["pct"] >= 0.65     # Next 20%：边缘/上下文
    ],
    ["Tier1_keep", "Tier2_review"],
    default="Tier3_drop"
)


In [46]:
scored.groupby(["dept","tier"]).size().unstack(fill_value=0)

tier,Tier1_keep,Tier2_review,Tier3_drop
dept,,,
doe,40,52,170
doi,62,81,265
epa,22,28,92
fema,19,25,80
hud,6,6,22
usda,41,54,174


In [47]:
# 保存最终结果
scored.to_csv(scores_csv, index=False)
scored

,dept,base,txt_path,meta_path,xml_path,txt_size,core_all,core_lead,climate_all,climate_lead,procedural,meta_hits,raw,score,score_norm,pct,tier
1,doi,01_03_2013_2012-30634,E:\2026_capstone\policy_data\doi_data\raw_text...,E:\2026_capstone\policy_data\doi_data\meta\01_...,E:\2026_capstone\policy_data\doi_data\xml\01_0...,1238801,1054,0,196,0,0,0,591.970058,1.000000,0.668842,0.995098,Tier1_keep
12,doe,01_05_2017_2016-30004,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\01_...,E:\2026_capstone\policy_data\doe_data\xml\01_0...,1072235,5,0,1004,3,0,0,214.750804,1.000000,0.941995,0.992366,Tier1_keep
58,doi,01_15_2025_2025-00709,E:\2026_capstone\policy_data\doi_data\raw_text...,E:\2026_capstone\policy_data\doi_data\meta\01_...,E:\2026_capstone\policy_data\doi_data\xml\01_1...,153525,221,5,1,0,0,1,128.549111,1.000000,0.143652,0.943627,Tier1_keep
68,epa,01_18_2023_2022-28595,E:\2026_capstone\policy_data\epa_data\raw_text...,E:\2026_capstone\policy_data\epa_data\meta\01_...,E:\2026_capstone\policy_data\epa_data\xml\01_1...,1144916,231,0,151,0,1,0,151.692358,1.000000,0.235459,0.957746,Tier1_keep
72,usda,01_19_2021_2021-00967,E:\2026_capstone\policy_data\usda_data\raw_tex...,E:\2026_capstone\policy_data\usda_data\meta\01...,E:\2026_capstone\policy_data\usda_data\xml\01_...,673689,1109,0,15,0,3,0,592.027565,1.000000,1.000000,1.000000,Tier1_keep
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
674,usda,07_19_2006_E6-11431,E:\2026_capstone\policy_data\usda_data\raw_tex...,E:\2026_capstone\policy_data\usda_data\meta\07...,E:\2026_capstone\policy_data\usda_data\xml\07_...,31838,0,0,2,0,0,0,0.477847,0.539737,0.000000,0.003717,Tier3_drop
192,doe,03_04_2011_2011-4878,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\03_...,E:\2026_capstone\policy_data\doe_data\xml\03_0...,16640,2,0,1,0,1,0,0.245097,0.520413,0.001075,0.007634,Tier3_drop
181,doe,02_29_2024_2024-04268,E:\2026_capstone\policy_data\doe_data\raw_text...,E:\2026_capstone\policy_data\doe_data\meta\02_...,E:\2026_capstone\policy_data\doe_data\xml\02_2...,8441,2,0,0,0,1,0,0.000000,0.500000,0.000000,0.003817,Tier3_drop
1037,fema,11_08_2019_2019-24372,E:\2026_capstone\policy_data\fema_data\raw_tex...,E:\2026_capstone\policy_data\fema_data\meta\11...,E:\2026_capstone\policy_data\fema_data\xml\11_...,9857,1,0,1,0,1,0,-0.375505,0.468749,0.000048,0.016129,Tier3_drop


In [54]:
# 将第三类文件移动到trash文件夹
BASE_OUT = Path("PIPLINE_OUTPUT") / "organized"

TIERS = ["Tier1_keep", "Tier2_review", "Tier3_trash"]

for tier in TIERS:
    (BASE_OUT / tier).mkdir(parents=True, exist_ok=True)


In [55]:
scored["tier_out"] = scored["tier"].replace({
    "Tier3_drop": "Tier3_trash"
})

In [56]:
def safe_move(src: Path, dst: Path, dry_run=True):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dry_run:
        return f"[DRY] {src} -> {dst}"
    shutil.move(str(src), str(dst))
    return f"MOVED {src.name}"

In [59]:
logs = []

for _, row in scored.iterrows():
    if row["tier_out"] != "Tier3_trash":
        continue

    dept = row["dept"]
    txt_path = Path(row["txt_path"])
    meta_path = Path(row["meta_path"]) if pd.notna(row["meta_path"]) else None

    txt_dst = BASE_OUT / "Tier3_trash" / f"{dept}_data" / "raw_text" / txt_path.name
    logs.append(safe_move(txt_path, txt_dst, dry_run=False))

    if meta_path and meta_path.exists():
        meta_dst = BASE_OUT / "Tier3_trash" / f"{dept}_data" / "meta" / meta_path.name
        logs.append(safe_move(meta_path, meta_dst, dry_run=False))

print(f"Moved {len(logs)} files to Tier3_trash")


Moved 1606 files to Tier3_trash
